# 13_Massive_ML_Variation_Sweep
## Materia Arche V3.2
### Hyperparameter sweep across sklearn regressors

Now that Phase 2 is triggered on the ML baseline (tau-b 0.249),
let's see if we can push the baseline higher before lab validation.

Models: RandomForest, GradientBoosting, ExtraTrees, AdaBoost
(XGBoost and LightGBM excluded — require libomp not available on this system)

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               ExtraTreesRegressor, AdaBoostRegressor)
from sklearn.model_selection import RandomizedSearchCV, train_test_split, KFold
from sklearn.metrics import make_scorer, mean_absolute_error
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded — ML variation sweep (sklearn only)")

Libraries loaded — ML variation sweep (sklearn only)


In [2]:
# Load data with correct column names
df = pd.read_csv("perovskite_stability_clean.csv")
print(f"Loaded {len(df)} devices")

ml_features = ['Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
               'MA', 'FA', 'Cs',
               'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
               'Perovskite_thickness', 'Cell_area_measured', 'JV_default_Voc',
               'JV_default_Jsc', 'JV_default_FF']

X = df[ml_features].fillna(0)
y = np.log1p(df['Stability_PCE_T80'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Features: {len(ml_features)}")

# Current baseline
rf_baseline = RandomForestRegressor(n_estimators=200, random_state=42)
rf_baseline.fit(X_train, y_train)
pred_baseline = rf_baseline.predict(X_test)
tau_baseline, _ = kendalltau(y_test, pred_baseline)
mae_baseline = mean_absolute_error(y_test, pred_baseline)
print(f"\nCurrent baseline (RF n=200): tau-b = {tau_baseline:.4f}, MAE = {mae_baseline:.4f}")

Loaded 1543 devices
Train: 1234, Test: 309
Features: 16



Current baseline (RF n=200): tau-b = 0.2490, MAE = 1.3053


## 1. Kendall tau-b scorer for hyperparameter search

In [3]:
def kendall_scorer(y_true, y_pred):
    tau, _ = kendalltau(y_true, y_pred)
    return tau if not np.isnan(tau) else 0.0

scorer = make_scorer(kendall_scorer, greater_is_better=True)
print("✅ Kendall tau-b scorer ready")

✅ Kendall tau-b scorer ready


## 2. Model sweep (30 random configs x 4 sklearn models)

In [4]:
models = {
    "RandomForest": (
        RandomForestRegressor(random_state=42),
        {
            'n_estimators': [100, 200, 300, 500],
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', None]
        }
    ),
    "GradientBoosting": (
        GradientBoostingRegressor(random_state=42),
        {
            'n_estimators': [100, 200, 300],
            'max_depth': [3, 5, 7, 10],
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'min_samples_split': [2, 5, 10],
            'subsample': [0.8, 1.0]
        }
    ),
    "ExtraTrees": (
        ExtraTreesRegressor(random_state=42),
        {
            'n_estimators': [100, 200, 300, 500],
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', None]
        }
    ),
    "AdaBoost": (
        AdaBoostRegressor(random_state=42),
        {
            'n_estimators': [50, 100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1, 0.5, 1.0],
            'loss': ['linear', 'square', 'exponential']
        }
    )
}

results = []

for name, (model, param_grid) in models.items():
    print(f"\nSweeping {name}...")
    search = RandomizedSearchCV(
        model, param_grid, n_iter=30, cv=5, scoring=scorer,
        random_state=42, n_jobs=-1
    )
    search.fit(X_train, y_train)
    
    pred = search.predict(X_test)
    tau_test, p_test = kendalltau(y_test, pred)
    mae_test = mean_absolute_error(y_test, pred)
    
    results.append({
        "Model": name,
        "Test_Tau_b": round(tau_test, 4),
        "Test_MAE": round(mae_test, 4),
        "CV_Tau_b": round(search.best_score_, 4),
        "Lift_vs_baseline": round(tau_test - tau_baseline, 4),
        "Best_Params": str(search.best_params_)
    })
    print(f"  Best CV tau-b: {search.best_score_:.4f}")
    print(f"  Test tau-b: {tau_test:.4f} (lift: {tau_test - tau_baseline:+.4f})")
    print(f"  Test MAE: {mae_test:.4f}")
    print(f"  Best params: {search.best_params_}")


Sweeping RandomForest...


  Best CV tau-b: 0.2602
  Test tau-b: 0.2629 (lift: +0.0140)
  Test MAE: 1.2899
  Best params: {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}

Sweeping GradientBoosting...


  Best CV tau-b: 0.2341
  Test tau-b: 0.2466 (lift: -0.0024)
  Test MAE: 1.3078
  Best params: {'subsample': 0.8, 'n_estimators': 200, 'min_samples_split': 2, 'max_depth': 10, 'learning_rate': 0.01}

Sweeping ExtraTrees...


  Best CV tau-b: 0.2844
  Test tau-b: 0.2675 (lift: +0.0186)
  Test MAE: 1.2570
  Best params: {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}

Sweeping AdaBoost...


  Best CV tau-b: 0.1415
  Test tau-b: 0.1741 (lift: -0.0749)
  Test MAE: 1.5288
  Best params: {'n_estimators': 200, 'loss': 'exponential', 'learning_rate': 0.1}


## 3. Leaderboard

In [5]:
leaderboard = pd.DataFrame(results).sort_values("Test_Tau_b", ascending=False)

print("=" * 75)
print("ML VARIATION LEADERBOARD")
print("=" * 75)
print(f"\nBaseline (RF n=200): tau-b = {tau_baseline:.4f}, MAE = {mae_baseline:.4f}")
print("")
print(leaderboard[['Model', 'Test_Tau_b', 'Test_MAE', 'CV_Tau_b', 'Lift_vs_baseline']].to_string(index=False))

best = leaderboard.iloc[0]
print(f"\nBest model: {best['Model']}")
print(f"  Test tau-b: {best['Test_Tau_b']}")
print(f"  Lift vs baseline: {best['Lift_vs_baseline']:+.4f}")
print(f"  Best params: {best['Best_Params']}")

if best['Lift_vs_baseline'] > 0:
    print(f"\nNew best model improves on baseline by {best['Lift_vs_baseline']:+.4f} tau-b")
else:
    print(f"\nBaseline RF (n=200) remains the best model.")

leaderboard.to_csv("ML_Variation_Leaderboard.csv", index=False)
print("\n✅ ML_Variation_Leaderboard.csv exported")

ML VARIATION LEADERBOARD

Baseline (RF n=200): tau-b = 0.2490, MAE = 1.3053

           Model  Test_Tau_b  Test_MAE  CV_Tau_b  Lift_vs_baseline
      ExtraTrees      0.2675    1.2570    0.2844            0.0186
    RandomForest      0.2629    1.2899    0.2602            0.0140
GradientBoosting      0.2466    1.3078    0.2341           -0.0024
        AdaBoost      0.1741    1.5288    0.1415           -0.0749

Best model: ExtraTrees
  Test tau-b: 0.2675
  Lift vs baseline: +0.0186
  Best params: {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}

New best model improves on baseline by +0.0186 tau-b

✅ ML_Variation_Leaderboard.csv exported


## 4. Cross-validate best model vs baseline

In [6]:
# Cross-validate baseline and best model on full data
import ast

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Baseline RF (n=200, defaults)
cv_taus_baseline = []
for train_idx, val_idx in kf.split(X):
    rf_cv = RandomForestRegressor(n_estimators=200, random_state=42)
    rf_cv.fit(X.iloc[train_idx], y.iloc[train_idx])
    pred_cv = rf_cv.predict(X.iloc[val_idx])
    tau_cv, _ = kendalltau(y.iloc[val_idx], pred_cv)
    cv_taus_baseline.append(tau_cv)

print(f"Baseline CV tau-b: {np.mean(cv_taus_baseline):.4f} +/- {np.std(cv_taus_baseline):.4f}")

# Best model from sweep — reconstruct with best params
best_name = best['Model']
best_params = ast.literal_eval(best['Best_Params'])

model_classes = {
    "RandomForest": RandomForestRegressor,
    "GradientBoosting": GradientBoostingRegressor,
    "ExtraTrees": ExtraTreesRegressor,
    "AdaBoost": AdaBoostRegressor
}

cv_taus_best = []
for train_idx, val_idx in kf.split(X):
    best_cv = model_classes[best_name](random_state=42, **best_params)
    best_cv.fit(X.iloc[train_idx], y.iloc[train_idx])
    pred_cv = best_cv.predict(X.iloc[val_idx])
    tau_cv, _ = kendalltau(y.iloc[val_idx], pred_cv)
    cv_taus_best.append(tau_cv)

print(f"Best ({best_name}) CV tau-b: {np.mean(cv_taus_best):.4f} +/- {np.std(cv_taus_best):.4f}")

cv_lift = np.mean(cv_taus_best) - np.mean(cv_taus_baseline)
print(f"CV lift: {cv_lift:+.4f}")

Baseline CV tau-b: 0.2569 +/- 0.0125


Best (ExtraTrees) CV tau-b: 0.2871 +/- 0.0140
CV lift: +0.0302


In [7]:
print("=" * 65)
print("SUMMARY")
print("=" * 65)
print(f"")
print(f"Models tested: {len(models)} (RF, GBR, ExtraTrees, AdaBoost)")
print(f"Configurations tested: ~120 (30 per model x 4 models)")
print(f"")
print(f"Baseline: RF (n=200) tau-b = {tau_baseline:.4f}")
print(f"Best sweep: {best['Model']} tau-b = {best['Test_Tau_b']}")
print(f"Lift: {best['Lift_vs_baseline']:+.4f}")
print(f"")
print(f"CV comparison:")
print(f"  Baseline: {np.mean(cv_taus_baseline):.4f} +/- {np.std(cv_taus_baseline):.4f}")
print(f"  Best:     {np.mean(cv_taus_best):.4f} +/- {np.std(cv_taus_best):.4f}")
print(f"")
if cv_lift > 0.01:
    print(f"The sweep found a meaningfully better model.")
    print(f"Consider using {best['Model']} for Phase 2 candidates.")
elif cv_lift > 0:
    print(f"Marginal improvement. Baseline RF is fine for Phase 2.")
else:
    print(f"No improvement. Baseline RF (n=200) remains the best choice.")
print(f"")
print(f"Phase 2 status: ACTIVE")
print(f"Milestones: 5/5 (ML-focused)")

SUMMARY

Models tested: 4 (RF, GBR, ExtraTrees, AdaBoost)
Configurations tested: ~120 (30 per model x 4 models)

Baseline: RF (n=200) tau-b = 0.2490
Best sweep: ExtraTrees tau-b = 0.2675
Lift: +0.0186

CV comparison:
  Baseline: 0.2569 +/- 0.0125
  Best:     0.2871 +/- 0.0140

The sweep found a meaningfully better model.
Consider using ExtraTrees for Phase 2 candidates.

Phase 2 status: ACTIVE
Milestones: 5/5 (ML-focused)
